In [1]:
import requests
import time
import pandas as pd
import csv
import re
import os
from datetime import datetime
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from dotenv import load_dotenv
load_dotenv()
guardian_api_token = os.getenv('guardian_TOKEN') 
# nyt_api_token = os.getenv('nyt_TOKEN')


c:\Users\hmaxw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_content_from_guardian(searches,start_date,end_date,api_key):
    content = []
    #is it 50 articels per page, so we go through mutliple pages that match with the same date needed
    for i in searches:
        page = 1
        while True:
            query = requests.get('https://content.guardianapis.com/search',params={'q' :i,'from-date':start_date,'to-date': end_date,'api-key': api_key,'page-size'  : 50,'page'       : page,
                                                                                    'show-fields': 'headline,body'}
                                                                                    )
            data = query.json()['response']
            content.extend(data['results'])
            if page >= data['pages']:
                break
            page += 1
    return content
#fields is a dict of title and content

searches = ['(Iran OR Israel) AND oil AND war',
    '"oil price" AND Iran',
    '"energy market" AND Iran',
    '"oil volatility" AND Iran',
    '"Strait of Hormuz"',
    'Iran AND sanctions AND oil',
    'energy volatility',
    'investor sentiment',
    '"market volatility" AND energy',]

raw_guardian_df = pd.DataFrame(get_content_from_guardian(searches, '2025-06-01',str(datetime.today().date()),guardian_api_token))

In [18]:
guardian_df = raw_guardian_df.copy()

In [19]:
guardian_df['date'] = pd.to_datetime(guardian_df['webPublicationDate']).dt.strftime('%Y%m%d')
guardian_df.drop_duplicates(subset=['webUrl'], inplace=True)

In [20]:
guardian_df

,id,type,sectionId,sectionName,webPublicationDate,webTitle,webUrl,apiUrl,fields,isHosted,pillarId,pillarName,date,headline,body_text
0,business/2026/mar/19/oil-prices-gas-prices-ris...,article,business,Business,2026-03-19T14:21:54Z,Oil and gas prices jump after Iran and Israel ...,https://www.theguardian.com/business/2026/mar/...,https://content.guardianapis.com/business/2026...,{'headline': 'Oil and gas prices jump after Ir...,False,pillar/news,News,20260319,Oil and gas prices jump after Iran and Israel ...,Gas prices jumped to four-year highs and oil p...
1,world/2026/mar/02/us-israel-war-on-iran-dramat...,article,world,World news,2026-03-02T18:18:10Z,US-Israel war on Iran dramatically expands acr...,https://www.theguardian.com/world/2026/mar/02/...,https://content.guardianapis.com/world/2026/ma...,{'headline': 'US-Israel war on Iran dramatical...,False,pillar/news,News,20260302,US-Israel war on Iran dramatically expands acr...,The war in the Middle East triggered by the jo...
2,business/2026/mar/20/stock-market-dip-iran-war,article,business,Business,2026-03-20T20:12:14Z,US stock markets dip for fourth straight week ...,https://www.theguardian.com/business/2026/mar/...,https://content.guardianapis.com/business/2026...,{'headline': 'US stock markets dip for fourth ...,False,pillar/news,News,20260320,US stock markets dip for fourth straight week ...,"US stock markets dropped again on Friday, capp..."
3,business/2026/mar/26/markets-slump-us-israel-w...,article,business,Business,2026-03-26T21:01:18Z,US markets see biggest slump since start of US...,https://www.theguardian.com/business/2026/mar/...,https://content.guardianapis.com/business/2026...,{'headline': 'US markets see biggest slump sin...,False,pillar/news,News,20260326,US markets see biggest slump since start of US...,US markets saw their biggest slump since the s...
4,commentisfree/2026/apr/02/iran-war-expenses-cu...,article,commentisfree,Opinion,2026-04-02T11:00:22Z,The US-Israel war on Iran is accelerating de-d...,https://www.theguardian.com/commentisfree/2026...,https://content.guardianapis.com/commentisfree...,{'headline': 'The US-Israel war on Iran is acc...,False,pillar/opinion,Opinion,20260402,The US-Israel war on Iran is accelerating de-d...,The US-Israel war on Iran is expensive. It’s e...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6014,australia-news/live/2026/feb/10/sydney-protest...,liveblog,australia-news,Australia news,2026-02-10T08:15:54Z,Protesters chant ‘we have the right to demonst...,https://www.theguardian.com/australia-news/liv...,https://content.guardianapis.com/australia-new...,{'headline': 'Protesters chant ‘we have the ri...,False,pillar/news,News,20260210,None,None
6016,us-news/live/2025/jun/17/donald-trump-g7-iran-...,liveblog,us-news,US news,2025-06-18T02:08:55Z,New York mayoral candidate Brad Lander release...,https://www.theguardian.com/us-news/live/2025/...,https://content.guardianapis.com/us-news/live/...,{'headline': 'New York mayoral candidate Brad ...,False,pillar/news,News,20250618,None,None
6017,politics/live/2025/jun/20/assisted-dying-vote-...,liveblog,politics,Politics,2025-06-20T14:58:12Z,MPs back legalising assisted dying in England ...,https://www.theguardian.com/politics/live/2025...,https://content.guardianapis.com/politics/live...,{'headline': 'MPs back legalising assisted dyi...,False,pillar/news,News,20250620,None,None
6021,us-news/live/2025/jun/12/la-protests-los-angel...,liveblog,us-news,US news,2025-06-13T02:33:21Z,Newsom says use of national guard for Ice raid...,https://www.theguardian.com/us-news/live/2025/...,https://content.guardianapis.com/us-news/live/...,{'headline': 'Newsom says use of national guar...,False,pillar/news,News,20250613,None,None


In [13]:
guardian_df['headline'] = None
guardian_df['body_text'] = None

for i, item in enumerate(guardian_df['fields']):
    guardian_df.at[i, 'headline'] = item['headline']
    soup = BeautifulSoup(item['body'])
    body_text = soup.get_text()
    guardian_df.at[i, 'body_text'] = body_text

TypeError: 'float' object is not subscriptable

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode('oil price energy markets Iran war investor sentiment volatility crude supply',convert_to_tensor=True)

def relevancy(headline):
    temp = model.encode(headline, convert_to_tensor=True)
    return util.cos_sim(temp,embeddings).item()

guardian_df['relevancy'] = guardian_df['headline'].apply(relevancy)
guardian_df = guardian_df[guardian_df['relevancy'] > 0.38]

    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 27066.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TypeError: 'NoneType' object is not subscriptable

In [ ]:
guardian_df = guardian_df[['date','headline','body_text']]
guardian_df.to_csv('guardian_data_since_20250601.csv', index=False, encoding='utf-8')


In [ ]:
for i in guardian_df['headline']:
    print(i)

20260319
20260302
20260320
20260326
20260402
20260322
20260305
20260308
20260329
20260310
20260401
20260311
20260309
20260306
20260303
20260314
20260405
20260306
20260302
20260312
20260320
20260306
20260302
20260316
20260321
20260317
20260326
20260316
20260331
20260321
20260312
20260310
20260302
20260309
20260302
20260319
20260301
20260311
20260402
20260406
20260402
20260310
20260308
20260308
20260401
20260331
20260317
20260310
20260404
20260305
20260402
20260320
20260406
20260311
20260308
20260320
20260302
20260312
20260324
20260330
20260308
20260322
20260312
20260312
20260330
20260320
20260314
20260310
20260303
20260323
20260327
20260312
20260315
20260315
20260313
20260319
20260310
20260312
20260325
20260311
20260316
20260310
20260314
20260326
20260317
20260302
20260323
20260402
20260405
20260325
20260309
20260310
20260321
20260323
20260402
20260312
20260401
20260309
20260402
20260314
20260302
20260312
20260322
20260312
20260321
20260316
20260312
20260402
20260321
20260320
20260306
2

In [ ]:
# def get_content_from_nyt(input,start_date,end_date,api_key):
#     content = []
#     page = 0
#     while True:
#         time.sleep(7)  #max 10 queries per min
#         query = requests.get('https://api.nytimes.com/svc/search/v2/articlesearch.json',params={'q' :input,'begin_date':start_date,'end_date': end_date,'api-key': api_key,'page': page})
#         data = query.json()['response']
#         content.extend(data['docs'])
#         if len(data['docs']) < 10:  # NYT returns 10 per page
#             break
#         page += 1
#     return (content)

# nyt_df = pd.DataFrame(get_content_from_nyt('Iran Israel oil energy', '20250601',(datetime.today().date()).strftime('%Y%m%d'),nyt_api_token))
# nyt_df = nyt_df[['pub_date','headline','abstract']]

In [ ]:
# nyt_df.to_csv('nyt_data_since_20250601.csv', index=False, encoding='utf-8')